# Chat with local Gemma 3n

A scratchpad for raw interaction with the locally-loaded MLX model —
no JSON parsing, no schemas, no repair pipeline. Just prompt → response.

Useful for:
- Eyeballing whether the model actually works at all
- Seeing what it returns when you ask for structured output (understanding why the repair layer fires)
- Testing image understanding

Run cells top to bottom. The model loads on first inference (cached thereafter for the lifetime of the kernel).

In [ ]:
# Make the repo root importable. The notebook lives in notebooks/, so go up one.
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force the local provider regardless of what .env says.
os.environ["LLM_PROVIDER"] = "local"

from src.llm import get_model, active_model_name
print(f"Using: {active_model_name()}")

In [ ]:
# Helper: send a raw prompt (+ optional images) and return the raw generated text.
# No JSON parsing, no schema — whatever the model produces is what you see.

def ask(prompt: str, images=None, max_tokens: int = 512, verbose: bool = False) -> str:
    """Send a prompt to local Gemma 3n and return the raw text response.

    - `prompt`: the user message. System prompt is empty here — add instructions inline
      if you want to mimic a structured-output call.
    - `images`: optional list of file paths (str/Path) or PIL.Image instances.
    - `max_tokens`: cap on generated tokens. 512 is plenty for chat-style replies.
    - `verbose`: if True, mlx-vlm prints per-token timing to stderr.
    """
    from mlx_vlm import generate
    from mlx_vlm.prompt_utils import apply_chat_template

    model, processor, config = get_model()

    # Normalise images: mlx-vlm's apply_chat_template wants a num_images count.
    imgs = images or []

    formatted = apply_chat_template(processor, config, prompt, num_images=len(imgs))
    out = generate(
        model,
        processor,
        formatted,
        imgs if imgs else None,
        max_tokens=max_tokens,
        verbose=verbose,
    )
    # mlx-vlm returns either a str or a GenerateResult — normalise to text.
    return getattr(out, "text", out) if not isinstance(out, str) else out

## Say hi

In [ ]:
reply = ask("Hey, how are you? Also, what model are you?")
print(reply)

## Something more involved

In [ ]:
reply = ask("Explain quantization in one paragraph, like I'm a Python developer who hasn't used ML much.")
print(reply)

## Stress-test the structured-output prompt

Few-shot prompt aligned with the actual `FileSummary` schema. Field names in
the example must match the schema exactly — small models copy the structure
they see, so `description` → bug, `summary` → correct.

In [ ]:
newprompt = """You are a file analyzer. Given a file's content, output a JSON object describing what the file is and the details someone might use to find it later via keyword search.

The JSON MUST have exactly these keys (and only these keys):
- title (string, <=80 chars)
- summary (string, 3-7 sentences of natural prose)
- content_type (one of: \"image\", \"pdf\", \"docx\", \"xlsx\", \"text\", \"code\", \"markdown\", \"other\")
- keywords (list of 3-10 topical words)
- key_entities (list of named entities: people, organisations, places, products, branches — copied verbatim from the file)
- identifiers (list of exact tokens that uniquely distinguish this file: numeric IDs, dates in their ORIGINAL format, SKUs, version strings, exact prices with currency, URLs — copied verbatim)

EXAMPLE:
Input:
Filename: flight-receipt.pdf
Content type: pdf
Delta Airlines — Flight Receipt
Passenger: Jane Doe
Flight DL1492, Atlanta ATL -> Hartford BDL, 25 May 2022
Confirmation code: ABC123
Total charged: $247.50

Output:
{\"title\": \"Delta flight DL1492 Atlanta to Hartford — Jane Doe\", \"summary\": \"Delta Airlines flight receipt for passenger Jane Doe. Flight DL1492 from Atlanta ATL to Hartford BDL on 25 May 2022. Confirmation code ABC123. Total charged: $247.50.\", \"content_type\": \"pdf\", \"keywords\": [\"flight\", \"receipt\", \"airline\", \"delta\", \"travel\"], \"key_entities\": [\"Delta Airlines\", \"Jane Doe\", \"Atlanta ATL\", \"Hartford BDL\"], \"identifiers\": [\"DL1492\", \"25 May 2022\", \"ABC123\", \"$247.50\"]}

Now analyze the file below. Return ONLY the JSON object — no markdown fences, no code blocks, no commentary. Start with { and end with }.
"""

test_file_content = """Filename: local-test.md
Summarize this file.

Content type: markdown

---
# Local MLX Smoke File

Unique identifier: **LOCAL-SMOKE-9000**.
This file mentions the fictional **Obsidian Tribunal** as a topical anchor.
Date of test: 2026-04-16.
"""

combined = newprompt + "\n\n" + test_file_content

reply = ask(combined, max_tokens=2000)
print(reply)

## With an image

Point it at any PNG/JPG under `Test Content/` (or anywhere else on disk).

In [ ]:
# Find an image under Test Content/, if any exist.
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
img_path = next(
    (p for p in (REPO_ROOT / "Test Content").rglob("*") if p.suffix.lower() in IMAGE_EXTS),
    None,
)

if img_path is None:
    print("no image found under Test Content/ — drop one in and re-run this cell")
else:
    print(f"using: {img_path.relative_to(REPO_ROOT)}")
    reply = ask("What's in this image? Describe it in one sentence.", images=[str(img_path)])
    print("\n" + reply)